In [1]:
import pandas as pd
import requests
from tqdm import tqdm
import time
from pathlib import Path

LAB2_DATA_DIR = Path("siteChumakov") / "chumakov" / "static" / "data"
LAB2_DATA_DIR.mkdir(parents=True, exist_ok=True)

# ==================== НАСТРОЙКИ RAPIDAPI ====================
RAPID_API_KEY = "ff4af73c48mshffc8e8387a8b8b9p1fa660jsn3fbd61349d75"  # <--- Вставьте сюда свой ключ
RAPID_API_HOST = "meteostat.p.rapidapi.com"

# ==================== МЕТАДАННЫЕ (ЧЕРЕЗ RAPIDAPI) ====================
def get_station_metadata(station_id: str):
    url = "https://meteostat.p.rapidapi.com/stations/meta"
    querystring = {"id": station_id}
    headers = {
        "x-rapidapi-key": RAPID_API_KEY,
        "x-rapidapi-host": RAPID_API_HOST,
        "Content-Type": "application/json"
    }
    
    meta = {
        "station_id": station_id,
        "country": "TR",
        "station_name": None,
        "lat": None, "lon": None, "elevation": None
    }

    try:
        response = requests.get(url, headers=headers, params=querystring, timeout=10)
        if response.status_code == 200:
            data = response.json().get("data", {})
            # Если данных несколько, берем первый словарь
            if isinstance(data, list) and len(data) > 0:
                data = data[0]
            
            if data:
                meta["Name2"] = data.get("name", {}).get("en") # На RapidAPI чаще англ. названия
                meta["lat"] = data.get("location", {}).get("latitude")
                meta["lon"] = data.get("location", {}).get("longitude")
                meta["elevation"] = data.get("location", {}).get("elevation")
                meta["country"] = data.get("country")
                meta["region"] = data.get("region")
                meta["Name"] = data.get("identifier", {}).get("icao")
        else:
            print(f"Ошибка API {response.status_code} для {station_id}")
    except Exception as e:
        print(f"Ошибка при запросе к RapidAPI: {e}")
    
    return meta

# ==================== ПАРСИНГ ТАБЛИЦ (БЕЗ ЛИШНИХ СТРОК) ====================
def parse_climate_table(station_id: str, is_precip: bool = False):
    suffix = "_2" if is_precip else ""
    url = f"https://www.pogodaiklimat.ru/history/{station_id}{suffix}.htm"
    prefix = "Os" if is_precip else "Tem"

    try:
        # Используем lxml или html5lib для корректного чтения
        tables = pd.read_html(url)
        
        # Находим нужные таблицы. Обычно они идут парами: годы и значения
        years_df = tables[0].copy()
        years_df.columns = ["year"]
        
        data_df = tables[1].iloc[:, :12].copy()
        month_names = ["янв","фев","мар","апр","май","июн","июл","авг","сен","окт","ноя","дек"]
        data_df.columns = month_names

        df = pd.concat([years_df.reset_index(drop=True), data_df.reset_index(drop=True)], axis=1)

        # 1. ЧИСТКА ГОДА: превращаем в числа, всё что не число (текст в футере) станет NaN
        df["year"] = pd.to_numeric(df["year"], errors="coerce")
        # Удаляем строки, где год пустой (это и уберет лишние строки без данных)
        df = df.dropna(subset=["year"]).copy()
        df["year"] = df["year"].astype(int)

        # 2. ЧИСТКА ДАННЫХ: убираем ошибки ChainedAssignment
        for col in month_names:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            # Заменяем сервисные коды сайта на честный NaN
            df[col] = df[col].replace([999.9, -999.9, 999, 9999, -99.9], pd.NA)

        # Переименование колонок
        rename_dict = {m: f"{prefix}{i+1}" for i, m in enumerate(month_names)}
        return df.rename(columns=rename_dict)

    except Exception as e:
        print(f"Ошибка парсинга таблиц для {station_id}: {e}")
        return pd.DataFrame(columns=["year"])

# ==================== ЗАПУСК ====================
station_ids = ["17150", "17115", "17234", "17237", "17220", "17116", "17124", "17060", "17130", "17030", "17095", "17280", "17195", "17170", "17340"] 
all_data = []

print("🛰️ Подключаемся к Meteostat через RapidAPI...")

for sid in tqdm(station_ids):
    meta = get_station_metadata(sid)
    
    # Собираем климатические данные
    temp_df = parse_climate_table(sid, is_precip=False)
    precip_df = parse_climate_table(sid, is_precip=True)

    # Объединяем по году
    df_combined = pd.merge(temp_df, precip_df, on="year", how="outer")

    # Удаляем строки, где нет вообще никаких климатических данных (пустоты)
    val_cols = [c for c in df_combined.columns if c != 'year']
    df_combined = df_combined.dropna(subset=val_cols, how='all').copy()

    if df_combined.empty:
        continue

    # Пришиваем метаданные
    for k, v in meta.items():
        df_combined.loc[:, k] = v

    all_data.append(df_combined)
    time.sleep(1) # Соблюдаем лимиты RapidAPI

if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    
    # Организуем колонки в логичном порядке
    cols = ["station_id", "country", "region", "Name", "Name2", "lat", "lon", "elevation", "year"] + \
           [f"Tem{i}" for i in range(1,13)] + [f"Os{i}" for i in range(1,13)]
    
    final_df = final_df[[c for c in cols if c in final_df.columns]]
    final_df.to_csv(LAB2_DATA_DIR / "climate_data_rapidapi.csv", index=False, encoding="utf-8-sig")
    print(f"\n📊 Сбор окончен. Сохранено {len(final_df)} строк.")

🛰️ Подключаемся к Meteostat через RapidAPI...


100%|██████████| 15/15 [01:02<00:00,  4.18s/it]


📊 Сбор окончен. Сохранено 1375 строк.


In [2]:
import pandas as pd

# 1. Загрузка данных
df = pd.read_csv(LAB2_DATA_DIR / 'climate_data_rapidapi.csv')

# 2. Список колонок, которые остаются неизменными (идентификаторы)
id_vars = ['station_id', 'country', 'region', 'Name', 'Name2', 'lat', 'lon', 'elevation', 'year']

# 3. Преобразование из широкого формата в длинный
# stubnames=['Tem', 'Os'] — это префиксы столбцов, которые мы хотим собрать
# j='month' — имя новой колонки, куда попадут числа 1-12 из названий Tem1, Os1 и т.д.
df_long = pd.wide_to_long(
    df, 
    stubnames=['Tem', 'Os'], 
    i=id_vars, 
    j='month'
).reset_index()

# 4. (Опционально) Создание колонки с датой для удобства анализа
df_long['date'] = pd.to_datetime(df_long[['year', 'month']].assign(day=1))

# 5. Сортировка для наглядности
df_long = df_long.sort_values(['station_id', 'year', 'month'])

# Вывод результата
print(df_long.head(15))

# Сохранение в новый файл, если нужно
# Промежуточный df_long не сохраняем отдельным CSV: он полностью восстанавливается из climate_data_rapidapi.csv.

       station_id country region Name   Name2      lat      lon  elevation  \
10044       17030      TR    SAM  NaN  Samsun  41.2833  36.3333          4   
10045       17030      TR    SAM  NaN  Samsun  41.2833  36.3333          4   
10046       17030      TR    SAM  NaN  Samsun  41.2833  36.3333          4   
10047       17030      TR    SAM  NaN  Samsun  41.2833  36.3333          4   
10048       17030      TR    SAM  NaN  Samsun  41.2833  36.3333          4   
10049       17030      TR    SAM  NaN  Samsun  41.2833  36.3333          4   
10050       17030      TR    SAM  NaN  Samsun  41.2833  36.3333          4   
10051       17030      TR    SAM  NaN  Samsun  41.2833  36.3333          4   
10052       17030      TR    SAM  NaN  Samsun  41.2833  36.3333          4   
10053       17030      TR    SAM  NaN  Samsun  41.2833  36.3333          4   
10054       17030      TR    SAM  NaN  Samsun  41.2833  36.3333          4   
10055       17030      TR    SAM  NaN  Samsun  41.2833  36.3333 

## Расчет индексов засухи через библиотеку `spei`
- SPI3, SPI6, SPI9, SPI12: для станции 17150
- SPEI3, SPEI6, SPEI9, SPEI12: для всех станций
- SSFI и SGI: для станции 17150

> Примечание: в исходном наборе нет наблюдаемых рядов стока и грунтовых вод, поэтому SSFI/SGI ниже считаются как прокси на основе климатических рядов (осадки и водный баланс).

In [3]:
import pandas as pd
from spei import spi, spei, ssfi, sgi

# 1) Загружаем данные и приводим их к помесячному формату
df = pd.read_csv(LAB2_DATA_DIR / "climate_data_rapidapi.csv")

# Для Os: если отсутствует >= половины значений, строку удаляем; иначе заполняем средним по строке
os_cols = [c for c in df.columns if c.startswith('Os')]
if os_cols:
    missing_os = df[os_cols].isna().sum(axis=1)
    df = df.loc[missing_os < (len(os_cols) / 2)].copy()
    
    # Заполняем средним по каждому столбцу
    df.loc[:, os_cols] = df[os_cols].fillna(df[os_cols].mean())


id_vars = ["station_id", "country", "region", "Name", "Name2", "lat", "lon", "elevation", "year"]
df_long = pd.wide_to_long(df, stubnames=["Tem", "Os"], i=id_vars, j="month").reset_index()
df_long["date"] = pd.to_datetime(df_long[["year", "month"]].assign(day=1))
df_long = df_long.sort_values(["station_id", "date"])

# 2) Станция 17150: SPI(3/6/9/12), SPEI(3/6/9/12), SSFI и SGI (прокси)
station_17150 = df_long[df_long["station_id"].astype(str) == "17150"].copy()
station_17150 = station_17150.sort_values("date").set_index("date")

precip = pd.to_numeric(station_17150["Os"], errors="coerce")
temp = pd.to_numeric(station_17150["Tem"], errors="coerce")
common = pd.DataFrame({"P": precip, "T": temp}).dropna()

# Простой PET-прокси из температуры для водного баланса (только для демонстрации SPEI/SGI)
pet_proxy = common["T"].clip(lower=0) * 2.0
water_balance = common["P"] - pet_proxy

# Прокси-ряд стока и грунтовых вод, т.к. прямых наблюдений нет в исходном CSV
streamflow_proxy = common["P"].rolling(3, min_periods=1).mean()
groundwater_proxy = water_balance.rolling(12, min_periods=1).mean().cumsum()

indices_17150 = pd.DataFrame(index=common.index)
for scale in [3, 6, 9, 12]:
    indices_17150[f"SPI{scale}"] = spi(common["P"], timescale=scale)
    indices_17150[f"SPEI{scale}"] = spei(water_balance, timescale=scale)

# В библиотеке spei индекс стока называется SSFI
indices_17150["SSFI3"] = ssfi(streamflow_proxy, timescale=3)
indices_17150["SGI3"] = sgi(groundwater_proxy, timescale=3)
indices_17150 = indices_17150.reset_index().rename(columns={"index": "date"})
indices_17150.insert(0, "station_id", "17150")

# 3) Все станции: SPEI(3/6/9/12)
spei_all = []
for sid, grp in df_long.groupby("station_id", dropna=False):
    grp = grp.sort_values("date").set_index("date")
    p = pd.to_numeric(grp["Os"], errors="coerce")
    t = pd.to_numeric(grp["Tem"], errors="coerce")
    common_st = pd.DataFrame({"P": p, "T": t}).dropna()

    # Минимум 5 лет месячных наблюдений для более стабильной подгонки распределения
    if len(common_st) < 60:
        continue

    pet_st = common_st["T"].clip(lower=0) * 2.0
    wb_st = common_st["P"] - pet_st

    out = pd.DataFrame(index=common_st.index)
    out["station_id"] = str(sid)
    for scale in [3, 6, 9, 12]:
        out[f"SPEI{scale}"] = spei(wb_st, timescale=scale)
    spei_all.append(out.reset_index().rename(columns={"index": "date"}))

spei_all_stations = pd.concat(spei_all, ignore_index=True).sort_values(["station_id", "date"])

# 4) Сохраняем результаты
indices_17150.to_csv(LAB2_DATA_DIR / "indices_station_17150.csv", index=False, encoding="utf-8-sig")
spei_all_stations.to_csv(LAB2_DATA_DIR / "spei_all_stations.csv", index=False, encoding="utf-8-sig")

print("Готово.")
print(f"Станция 17150: {len(indices_17150)} строк -> siteChumakov/chumakov/static/data/indices_station_17150.csv")
print(f"Все станции (SPEI): {len(spei_all_stations)} строк -> siteChumakov/chumakov/static/data/spei_all_stations.csv")
print()
print(indices_17150.head(10))

Готово.
Станция 17150: 912 строк -> siteChumakov/chumakov/static/data/indices_station_17150.csv
Все станции (SPEI): 12680 строк -> siteChumakov/chumakov/static/data/spei_all_stations.csv

  station_id       date      SPI3     SPEI3      SPI6     SPEI6      SPI9  \
0      17150 1950-01-01       NaN       NaN       NaN       NaN       NaN   
1      17150 1950-02-01       NaN       NaN       NaN       NaN       NaN   
2      17150 1950-03-01 -0.171185 -0.113648       NaN       NaN       NaN   
3      17150 1950-04-01 -1.114770 -1.276930       NaN       NaN       NaN   
4      17150 1950-05-01 -0.209855 -0.285331       NaN       NaN       NaN   
5      17150 1950-06-01  0.100047  0.046618 -0.208080 -0.193810       NaN   
6      17150 1950-07-01  0.280291  0.260317 -0.806318 -0.961356       NaN   
7      17150 1950-08-01 -0.346671 -0.341034 -0.488974 -0.602901       NaN   
8      17150 1950-09-01  0.312018 -0.027332 -0.029241 -0.163627 -0.284093   
9      17150 1950-10-01 -0.147340  0.02636

In [7]:
import pandas as pd

# Один конкретный момент времени: одна дата для всех станций
# Для месячных данных дата соответствует первому числу месяца
target_date = pd.Timestamp("2021-08-01")

weather_at_moment = df_long.loc[
    df_long["date"].eq(target_date),
    [
        "station_id",
        "country",
        "region",
        "Name",
        "Name2",
        "lat",
        "lon",
        "elevation",
        "date",
        "Tem",
        "Os",
    ],
].copy()

# Добавляем SPEI для выбранной даты по всем станциям
spei_at_moment = spei_all_stations.loc[
    spei_all_stations["date"].eq(target_date),
    ["station_id", "date", "SPEI3", "SPEI6", "SPEI9", "SPEI12"],
].copy()

weather_at_moment["station_id"] = weather_at_moment["station_id"].astype(str)
spei_at_moment["station_id"] = spei_at_moment["station_id"].astype(str)

weather_at_moment = weather_at_moment.merge(
    spei_at_moment,
    on=["station_id", "date"],
    how="left",
).sort_values("station_id").reset_index(drop=True)

print(f"Найдено строк: {len(weather_at_moment)} для даты {target_date.date()}")
if weather_at_moment.empty:
    print("На эту дату данных нет. Замените target_date на дату, которая есть в df_long.")
else:
    display(weather_at_moment.head(20))
    # Снимок на одну дату нужен только для просмотра в ноутбуке, отдельный CSV для лабораторных не сохраняем.

weather_at_moment.to_csv("метеостанции_Чумаков.csv", index=False)

Найдено строк: 15 для даты 2021-08-01


,station_id,country,region,Name,Name2,lat,lon,elevation,date,Tem,Os,SPEI3,SPEI6,SPEI9,SPEI12
0,17030,TR,SAM,NaN,Samsun,41.2833,36.3333,4,2021-08-01,25.3,51.0,0.952129,1.097813,0.341754,-0.209012
1,17060,TR,IST,LTBA,Istanbul / Ataturk,40.9667,28.8167,48,2021-08-01,26.7,0.0,-0.953585,-0.890749,-1.078412,-1.015587
2,17095,TR,EZR,NaN,Erzurum Bolge / Dada?,39.9000,41.2833,1869,2021-08-01,20.4,63.0,0.275991,-1.581533,-1.531499,-1.604039
3,17115,TR,BAL,LTBG,Bandirma,40.3167,27.9667,51,2021-08-01,25.9,2.0,-0.899411,0.197676,-0.290466,-1.057690
4,17116,TR,BRS,LTBE,Bursa,40.1833,29.0667,101,2021-08-01,26.4,0.2,0.438939,0.330787,0.979853,0.357152
5,17124,TR,ESK,LTBI,Eskisehir,39.7833,30.5667,785,2021-08-01,22.8,0.0,0.827693,0.435281,-0.186842,0.192925
6,17130,TR,ANK,NaN,Ankara / Central,39.9500,32.8833,891,2021-08-01,25.6,17.0,-0.006947,-0.745601,-1.256350,-1.791397
7,17150,TR,BAL,LTBF,Balikesir,39.6167,27.9167,101,2021-08-01,26.3,0.0,0.477276,1.046468,1.034919,0.236271
8,17170,TR,VAN,LTCI,Van,38.4500,43.3167,1667,2021-08-01,22.4,1.0,-1.524821,-2.189848,-2.282761,-2.570915
9,17195,TR,KAY,LTAU,Kayseri / Erkilet,38.7833,35.4833,1053,2021-08-01,21.7,17.0,0.715751,0.535498,0.310831,-0.579774
